# Experiment 5 — Paragraph vs sentence: TF-IDF + surprisal and TF-IDF + MDL

Same models and features as Experiment 2, trained **separately at paragraph level and at sentence level**, so the two
levels can be compared directly. Does surprisal / MDL help more when the unit is one sentence instead of a paragraph?

| Feature set | Features |
|---|---|
| `tfidf_only` | TF-IDF (baseline) |
| `tfidf+surprisal` | TF-IDF + GPT-2 `mean_surprisal` |
| `tfidf+mdl` | TF-IDF + `mean_mdl` |
| `tfidf+surprisal+mdl` | TF-IDF + both |

Each feature set runs on all three master sheets (`absolute`, `cumulative`, `step`) of each level.

- same logistic regression, TF-IDF settings, random A/B swap and SEED as Experiment 2
- same 20% test articles at both levels (`data/test-articles.csv`)
- pairs with an undefined paragraph / sentence (e.g. one-word "Steps": no MDL) are left out, from each level's
  `feature-engineering/undefined_paragraphs.csv`
- sentence groups were split at weak links (BLEU <= 0.06) in `build-master-sheet`, so no pair compares unrelated sentences
- paragraph numbers must equal Experiment 2 exactly (checked below)

Output: `output/comparison/tables/exp5-paragraph-vs-sentence.csv` and `output/comparison/figures/exp5-paragraph-vs-sentence.png`

## 1. Settings

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score)
from sklearn.preprocessing import StandardScaler

LEVELS = ["paragraph", "sentence"]
FILES = ["absolute", "cumulative", "step"]
comparison_dir = Path("../../output/comparison")
results_path = comparison_dir / "tables" / "exp5-paragraph-vs-sentence.csv"
figure_path = comparison_dir / "figures" / "exp5-paragraph-vs-sentence.png"
results_path.parent.mkdir(parents=True, exist_ok=True)
figure_path.parent.mkdir(parents=True, exist_ok=True)

SEED = 42

# Each feature set = the numeric columns added next to TF-IDF (same as Experiment 2)
FEATURE_SETS = {
    "tfidf_only": [],
    "tfidf+surprisal": ["mean_surprisal"],
    "tfidf+mdl": ["mean_mdl"],
    "tfidf+surprisal+mdl": ["mean_surprisal", "mean_mdl"],
}
NUMERIC = ["mean_surprisal", "mean_mdl"]  # every numeric column we need to load

# one shared list of test articles for both levels (made by Experiment 1 from the paragraph split)
test_articles = set(pd.read_csv("../../data/test-articles.csv")["article"].astype(str))
print("test articles:", len(test_articles))

## 2. Helpers

Same as Experiment 2. `defined_pairs` takes the undefined keys of the level as an argument, because each level has its own list.

In [ ]:
def load_undefined_keys(master_dir):
    #paragraphs / sentences where a metric could not be calculated - made by feature-engineering.ipynb
    undefined = pd.read_csv(master_dir / "feature-engineering" / "undefined_paragraphs.csv")
    return set(zip(undefined["article"].astype(str), undefined["revision_group_no"], undefined["revision_depth"]))


def defined_pairs(df, base_depth_column, edit_depth_column, undefined_keys):
    #True for every pair where both versions have all metrics
    base_keys = zip(df["article"].astype(str), df["revision_group_no"], df[base_depth_column])
    edit_keys = zip(df["article"].astype(str), df["revision_group_no"], df[edit_depth_column])
    return np.array([base not in undefined_keys and edit not in undefined_keys for base, edit in zip(base_keys, edit_keys)])


def add_numeric(tfidf_train, tfidf_test, values_train, values_test):
    if values_train.shape[1] == 0:  # tfidf_only: nothing to add
        return tfidf_train, tfidf_test
    scaler = StandardScaler()  # mean 0, std 1 (learned on train only)
    values_train = scaler.fit_transform(values_train)
    values_test = scaler.transform(values_test)
    X_train = hstack([tfidf_train, csr_matrix(values_train)]).tocsr()
    X_test = hstack([tfidf_test, csr_matrix(values_test)]).tocsr()
    return X_train, X_test


def train_and_test(X_train, y_train, X_test, y_test):
    model = LogisticRegression(max_iter=2000, random_state=SEED)
    model.fit(X_train, y_train)
    predicted = model.predict(X_test)
    probability = model.predict_proba(X_test)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_test, predicted).ravel()
    return {
        "accuracy": accuracy_score(y_test, predicted),
        "precision": precision_score(y_test, predicted),
        "recall": recall_score(y_test, predicted),
        "f1": f1_score(y_test, predicted),
        "auc": roc_auc_score(y_test, probability),
        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
    }


def show(level, file, feature_set, m):
    print(f"{level:9s} {file:10s} {feature_set:22s} acc={m['accuracy']:.4f}  f1={m['f1']:.4f}  auc={m['auc']:.4f}  "
          f"[tn={m['tn']} fp={m['fp']} fn={m['fn']} tp={m['tp']}]")


results = []

## 3. Absolute file (one row = one version, label `edit` = 1, `base` = 0)

In [ ]:
for level in LEVELS:
    master_dir = Path(f"../../data/{level}/master")
    undefined_keys = load_undefined_keys(master_dir)
    df = pd.read_csv(
        master_dir / "master-absolute.csv",
        usecols=["article", "revision_group_no", "pair_older_depth", "pair_newer_depth", "label", "text", *NUMERIC],
        low_memory=False,
    )
    keep = defined_pairs(df, "pair_older_depth", "pair_newer_depth", undefined_keys)
    df = df[keep].reset_index(drop=True)
    text = df["text"].fillna("").astype(str)
    y = (df["label"] == "edit").astype(int).to_numpy()
    is_test = df["article"].astype(str).isin(test_articles).to_numpy()
    is_train = ~is_test
    print(f"\n=== {level} absolute ===  left out: {(~keep).sum()} rows  train: {is_train.sum()}  test: {is_test.sum()}")

    vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20000)
    tfidf_train = vectorizer.fit_transform(text[is_train])
    tfidf_test = vectorizer.transform(text[is_test])

    for feature_set, columns in FEATURE_SETS.items():
        values = df[columns].fillna(0).to_numpy(dtype=float)
        X_train, X_test = add_numeric(tfidf_train, tfidf_test, values[is_train], values[is_test])
        m = train_and_test(X_train, y[is_train], X_test, y[is_test])
        show(level, "absolute", feature_set, m)
        results.append({"level": level, "file": "absolute", "feature_set": feature_set,
                        "n_train": int(is_train.sum()), "n_test": int(is_test.sum()), **m})

## 4. Pair files (cumulative and step)

Each pair is shown as (A, B) in random order, label = **B is the edit**.
Text feature `TF-IDF(B) − TF-IDF(A)`, numeric features `delta_*` (edit − base), flipped when swapped.

In [ ]:
for level in LEVELS:
    master_dir = Path(f"../../data/{level}/master")
    undefined_keys = load_undefined_keys(master_dir)
    for setting in ["cumulative", "step"]:
        delta_cols = ["delta_" + name for name in NUMERIC]
        df = pd.read_csv(
            master_dir / f"master-pairs-{setting}.csv",
            usecols=["article", "revision_group_no", "base_depth", "edit_depth", "base_text", "edit_text", *delta_cols],
            low_memory=False,
        )
        keep = defined_pairs(df, "base_depth", "edit_depth", undefined_keys)
        df = df[keep].reset_index(drop=True)
        base_text = df["base_text"].fillna("").astype(str).to_numpy(dtype=object)
        edit_text = df["edit_text"].fillna("").astype(str).to_numpy(dtype=object)

        # swap = True -> A is the edit (label 0); swap = False -> B is the edit (label 1)
        swap = np.random.default_rng(SEED).integers(0, 2, size=len(df)).astype(bool)
        A = np.where(swap, edit_text, base_text)
        B = np.where(swap, base_text, edit_text)
        y = (~swap).astype(int)
        sign = np.where(swap, -1.0, 1.0)

        is_test = df["article"].astype(str).isin(test_articles).to_numpy()
        is_train = ~is_test
        print(f"\n=== {level} {setting} ===  left out: {(~keep).sum()} pairs  train: {is_train.sum()}  test: {is_test.sum()}")

        vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20000)
        vectorizer.fit(list(A[is_train]) + list(B[is_train]))
        tfidf_train = vectorizer.transform(B[is_train]) - vectorizer.transform(A[is_train])
        tfidf_test = vectorizer.transform(B[is_test]) - vectorizer.transform(A[is_test])

        for feature_set, columns in FEATURE_SETS.items():
            cols = ["delta_" + name for name in columns]
            values = df[cols].fillna(0).to_numpy(dtype=float) * sign[:, None]
            X_train, X_test = add_numeric(tfidf_train, tfidf_test, values[is_train], values[is_test])
            m = train_and_test(X_train, y[is_train], X_test, y[is_test])
            show(level, setting, feature_set, m)
            results.append({"level": level, "file": setting, "feature_set": feature_set,
                            "n_train": int(is_train.sum()), "n_test": int(is_test.sum()), **m})

## 5. Results

In [ ]:
results_df = pd.DataFrame(results)
results_df.to_csv(results_path, index=False)
print("saved", results_path)

# CHECK - paragraph numbers must be exactly the ones from Experiment 2 (same data, features, seed)
exp2 = pd.read_csv("../../output/paragraph/tables/exp2-tfidf-surprisal-mdl.csv")
mine = results_df[results_df.level == "paragraph"].set_index(["file", "feature_set"])
theirs = exp2.set_index(["file", "feature_set"]).loc[mine.index]
for metric in ["accuracy", "precision", "recall", "f1", "auc", "tn", "fp", "fn", "tp"]:
    assert np.allclose(mine[metric].to_numpy(dtype=float), theirs[metric].to_numpy(dtype=float)), metric
print("paragraph results match Experiment 2")

In [ ]:
# Accuracy and AUC side by side: one row per feature set, one column per (level, file)
order = list(FEATURE_SETS)
accuracy_table = results_df.pivot(index="feature_set", columns=["level", "file"], values="accuracy").loc[order]
auc_table = results_df.pivot(index="feature_set", columns=["level", "file"], values="auc").loc[order]
print("ACCURACY"); print(accuracy_table.round(4).to_string())
print("\nAUC"); print(auc_table.round(4).to_string())

In [ ]:
# What does each feature add on top of TF-IDF only? (accuracy points, + = better than TF-IDF alone)
gain_table = (accuracy_table - accuracy_table.loc["tfidf_only"]).drop(index="tfidf_only") * 100
print("ACCURACY GAIN OVER TF-IDF ONLY (points)")
print(gain_table.round(2).to_string())

## 6. Plot

Top: accuracy of every feature set. Bottom: what surprisal / MDL add on top of TF-IDF only (accuracy points).
One column per master sheet, paragraph vs sentence side by side.

In [ ]:
PARAGRAPH, SENTENCE = "#2a78d6", "#eb6834"  # categorical slots 1 and 2 (colour-blind checked pair)
INK, INK_2, GRID = "#1f1f1f", "#5f5e5a", "#e6e5e1"
level_colors = {"paragraph": PARAGRAPH, "sentence": SENTENCE}
labels = {"tfidf_only": "TF-IDF only", "tfidf+surprisal": "+ surprisal", "tfidf+mdl": "+ MDL", "tfidf+surprisal+mdl": "+ both"}
width = 0.38

fig, axes = plt.subplots(2, len(FILES), figsize=(15, 8.6), gridspec_kw={"height_ratios": [1.25, 1]})
for col, file in enumerate(FILES):
    # top: accuracy of every feature set
    ax = axes[0, col]
    x = np.arange(len(order))
    for k, level in enumerate(LEVELS):
        values = accuracy_table[(level, file)].to_numpy()
        bars = ax.bar(x + (k - 0.5) * width, values, width=width - 0.03, color=level_colors[level], label=level, zorder=2)
        for bar, v in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width() / 2, v + 0.004, f"{v:.3f}", ha="center", va="bottom", fontsize=7.5, color=INK)
    low = accuracy_table.xs(file, axis=1, level="file").to_numpy().min()
    ax.set_ylim(max(0.45, low - 0.05), accuracy_table.to_numpy().max() + 0.03)
    ax.axhline(0.5, color="#9d9c97", lw=0.8, ls=":", zorder=1)  # chance
    ax.set_xticks(x, [labels[f] for f in order], fontsize=9)
    ax.set_title(f"{file}", loc="left", fontsize=11, weight="bold")
    if col == 0:
        ax.set_ylabel("Test accuracy")

    # bottom: gain over TF-IDF only
    ax = axes[1, col]
    gains = order[1:]
    x = np.arange(len(gains))
    for k, level in enumerate(LEVELS):
        values = gain_table[(level, file)].loc[gains].to_numpy()
        bars = ax.bar(x + (k - 0.5) * width, values, width=width - 0.03, color=level_colors[level], zorder=2)
        for bar, v in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width() / 2, v + (0.08 if v >= 0 else -0.08), f"{v:+.2f}",
                    ha="center", va="bottom" if v >= 0 else "top", fontsize=7.5, color=INK)
    span = np.abs(gain_table.to_numpy()).max()
    ax.set_ylim(min(-0.6, gain_table.to_numpy().min() - 0.4 * span / 4), span * 1.18 + 0.2)
    ax.axhline(0, color="#9d9c97", lw=0.9, zorder=1)
    ax.set_xticks(x, [labels[f] for f in gains], fontsize=9)
    if col == 0:
        ax.set_ylabel("Accuracy gain over TF-IDF only (points)")

for ax in axes.flat:
    ax.grid(axis="y", color=GRID, lw=0.8, zorder=0)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(colors=INK_2)

fig.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=level_colors[l]) for l in LEVELS], labels=["Paragraph", "Sentence"],
           loc="upper right", ncol=2, frameon=False, fontsize=10, bbox_to_anchor=(0.99, 0.995))
n = {l: int(results_df[(results_df.level == l) & (results_df.file == "step")].n_test.iloc[0]) for l in LEVELS}
fig.suptitle("Paragraph vs sentence · logistic regression on TF-IDF + surprisal / MDL", x=0.01, ha="left", fontsize=14, weight="bold")
fig.text(0.01, 0.935, f"Same 20% test articles at both levels · step test pairs: paragraph {n['paragraph']:,}, sentence {n['sentence']:,}"
         " · dotted line = chance (0.5)", fontsize=10, color=INK_2)
fig.tight_layout(rect=[0, 0, 1, 0.93])
fig.savefig(figure_path, dpi=200, facecolor="white")
plt.show()
print("saved", figure_path)